In [14]:
import osmnx as ox
import folium
import geopandas as gpd
import pandas as pd
from pathlib import Path
import duckdb

In [4]:
ox.settings.use_cache = True
ox.settings.overpass_settings = "[out:json][timeout:60]"
ox.settings.requests_timeout = 90

pois = ox.features_from_place(
    "Luzern, Switzerland",
    tags={"amenity": "school"}
)

print("Schools found:", len(pois))
display(pois[["name", "amenity", "geometry"]].head())

Schools found: 48


name amenity  \
element id                                                                  
node    304003296                  Pädagogische Hochschule Luzern  school   
        641801352                                        Felsberg  school   
        649041603                               Montessori Schule  school   
        2287991079  B.H.M.S. (Business Hotel & Management School)  school   
        2801774768                    EWTO WingTsun Schule Luzern  school   

                                    geometry  
element id                                    
node    304003296   POINT (8.33061 47.05656)  
        641801352   POINT (8.31372 47.05853)  
        649041603   POINT (8.31673 47.05637)  
        2287991079  POINT (8.29315 47.05344)  
        2801774768  POINT (8.31605 47.05633)

In [5]:
schools = pois.copy()

display(
    schools[["name", "amenity", "geometry"]]
    .sort_values("name")
)

name amenity  \
element  id                                                                   
way      194602359             Aus- und Weiterbildungszentrum G'ART  school   
node     2287991079   B.H.M.S. (Business Hotel & Management School)  school   
         8725697611           Berufsbildungszentrum Bau und Gewerbe  school   
         10606239342                         Bildungszentrum Luzern  school   
         3975639459                               Castle's language  school   
         2801774768                     EWTO WingTsun Schule Luzern  school   
         641801352                                         Felsberg  school   
         7252978485                                     Four-Forest  school   
         4209158694                    HSO Wirtschaftschule Schweiz  school   
way      104279025                   Kantonsschule Alpenquai Luzern  school   
         87220865                              Kantonsschule Musegg  school   
         94851953                              Kantonsschule Musegg  school   
         101768237                          Kantonsschule Reussbühl  school   
         86335353                 Mittelschulzentrum Hirschengraben  school   
node     649041603                                Montessori Schule  school   
way      46340090                                            Musegg  school   
relation 7513551                        Oberstufenzentrum Mariahilf  school   
way      101391304                            Primarschule Moosmatt  school   
node     304003296                   Pädagogische Hochschule Luzern  school   
way      315588600                               Schulhaus Felsberg  school   
         315732708                              Schulhaus Fluhmühle  school   
         119547856                           Schulhaus Geissenstein  school   
         315733412                               Schulhaus Grenzhof  school   
         105583384                              Schulhaus Hubelmatt  school   
         101433158                            Schulhaus Littau-Dorf  school   
         315732217                                 Schulhaus Maihof  school   
         114021844                                   Schulhaus Matt  school   
         122580617                               Schulhaus Ruopigen  school   
         315731252                              Schulhaus Rönnimoos  school   
         115851812                              Schulhaus St. Karli  school   
         122588940                               Schulhaus Staffeln  school   
         108358741                             Schulhaus Steinhof 1  school   
         108358758                             Schulhaus Steinhof 2  school   
         114670009                              Schulhaus Tribschen  school   
         108358743                            Schulhaus Unterlöchli  school   
         114670008                                Schulhaus Wartegg  school   
         285244580                             Schulhaus Würzenbach  school   
         88642416                           Sekundarschule Utenberg  school   
         127105037                                    Sälischulhaus  school   
         89865232                                    Villa Schröder  school   
node     9442245059                                 École française  school   
way      94201126                                               NaN  school   
         119547812                                              NaN  school   
         119547813                                              NaN  school   
         119547855                                              NaN  school   
         391906826                                              NaN  school   
         531921054                                              NaN  school   
         1109124493                                             NaN  school   

                                                               geometry  
element  id                           

In [6]:
print("Total features:", len(schools))
print("Missing names:", schools["name"].isna().sum())
print("Geometry types:")
print(schools.geometry.geom_type.value_counts())

Total features: 48
Missing names: 7
Geometry types:
Polygon    37
Point      11
Name: count, dtype: int64


In [10]:
m = folium.Map(
    location=[47.05, 8.31],
    zoom_start=13
)

for _, school in schools.iterrows():
    point = school.geometry.representative_point()

    folium.CircleMarker(
        location=[point.y, point.x],
        radius=5,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=str(school.get("name", "Unknown school"))
    ).add_to(m)

display(m)

In [11]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)

schools.to_parquet(
    "../data/luzern_schools_current.parquet"
)

In [15]:
DATA_DIR = Path.home() / "Datasets"

con = duckdb.connect()

luzern_detectors = con.sql("""
    SELECT *
    FROM read_csv(
        ?,
        header = true,
        delim = ',',
        quote = '"',
        all_varchar = true
    )
    WHERE lower(citycode) = 'luzern'
""", params=[str(DATA_DIR / "detectors.csv")]).df()

# Convert coordinates to numeric
for col in ["lat", "long"]:
    luzern_detectors[col] = pd.to_numeric(
        luzern_detectors[col],
        errors="coerce"
    )

luzern_detectors = luzern_detectors.dropna(
    subset=["lat", "long"]
)

print(f"Luzern detectors: {len(luzern_detectors)}")
display(luzern_detectors.head())

Luzern detectors: 158


,detid,length,pos,fclass,road,limit,citycode,lanes,linkid,long,lat
0,ig11FD116_D20,0.117622605755175,0.042476931131866,primary,Baselstrasse,0,luzern,1,91,8.297066,47.052141
1,ig11FD116_D15,0.084304613395923,0.043320630438901,primary,Baselstrasse,0,luzern,1,92,8.297038,47.052108
2,ig11FD116_D35,0.268815714824,0.22362392624425,primary,Baselstrasse,0,luzern,1,90,8.297114,47.052192
3,ig11FD116_D34,0.11348333835617,0.068332388524665,primary,Baselstrasse,0,luzern,1,89,8.297129,47.052223
4,ig11FD116_D30,0.097651051543157,0.034251934100282,primary,Gütschstrasse,50,luzern,1,83,8.298206,47.052388


In [16]:
detectors = luzern_detectors.copy()

detectors_geo = gpd.GeoDataFrame(
    detectors,
    geometry=gpd.points_from_xy(
        detectors["long"],
        detectors["lat"]
    ),
    crs="EPSG:4326"
)

m = folium.Map(
    location=[47.05, 8.31],
    zoom_start=13
)

# Schools (red)
for _, school in schools.iterrows():
    point = school.geometry.representative_point()

    folium.CircleMarker(
        [point.y, point.x],
        radius=5,
        color="red",
        fill=True,
        popup=str(school.get("name", "School"))
    ).add_to(m)

# Detectors (blue)
for _, detector in detectors_geo.iterrows():
    folium.CircleMarker(
        [detector.geometry.y, detector.geometry.x],
        radius=4,
        color="blue",
        fill=True,
        popup=str(detector["detid"])
    ).add_to(m)

display(m)

In [17]:
detectors_m = detectors_geo.to_crs("EPSG:2056")
schools_m = schools.to_crs("EPSG:2056")

nearest = gpd.sjoin_nearest(
    detectors_m,
    schools_m[["name", "geometry"]],
    how="left",
    distance_col="nearest_school_m"
)

display(
    nearest[
        ["detid", "road", "name", "nearest_school_m"]
    ].head(10)
)

,detid,road,name,nearest_school_m
0,ig11FD116_D20,Baselstrasse,B.H.M.S. (Business Hotel & Management School),330.444241
1,ig11FD116_D15,Baselstrasse,B.H.M.S. (Business Hotel & Management School),330.146715
2,ig11FD116_D35,Baselstrasse,B.H.M.S. (Business Hotel & Management School),331.334560
3,ig11FD116_D34,Baselstrasse,B.H.M.S. (Business Hotel & Management School),330.929981
4,ig11FD116_D30,Gütschstrasse,Musegg,369.909463
5,ig11FD116_D25,Gütschstrasse,Musegg,371.240368
6,ig11FD116_D33,Baselstrasse,Musegg,366.167501
7,ig11FD100_D31,Seebrücke,Bildungszentrum Luzern,105.145652
8,ig11FD100_D32,Seebrücke,Bildungszentrum Luzern,105.202183
9,ig11FD100_D33,Bahnhofplatz,Bildungszentrum Luzern,41.060379


In [18]:
nearest["school_proximity"] = pd.cut(
    nearest["nearest_school_m"],
    bins=[0, 300, 500, 1000, float("inf")],
    labels=[
        "0-300m",
        "300-500m",
        "500-1000m",
        "1000m+"
    ],
    include_lowest=True
)

print(nearest["school_proximity"].value_counts())

school_proximity
0-300m       124
300-500m      36
500-1000m      0
1000m+         0
Name: count, dtype: int64


In [19]:
print("Original detectors:", len(detectors_geo))
print("Nearest join rows:", len(nearest))
print("Unique detectors:", nearest["detid"].nunique())

display(
    nearest[
        nearest.duplicated("detid", keep=False)
    ][["detid", "name", "nearest_school_m"]]
)

Original detectors: 158
Nearest join rows: 160
Unique detectors: 158


,detid,name,nearest_school_m
110,ig11FD110_D13,Primarschule Moosmatt,328.061188
110,ig11FD110_D13,NaN,328.061188
118,ig11FD110_D30,Primarschule Moosmatt,324.539525
118,ig11FD110_D30,NaN,324.539525


In [20]:
# Prefer named schools when multiple matches have equal distances
nearest["has_name"] = nearest["name"].notna()

nearest = (
    nearest
    .sort_values(
        ["detid", "nearest_school_m", "has_name"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset="detid", keep="first")
    .drop(columns="has_name")
    .reset_index(drop=True)
)

print("Detectors:", len(nearest))
print("Unique detectors:", nearest["detid"].nunique())

Detectors: 158
Unique detectors: 158


In [21]:
nearest["school_proximity"] = pd.cut(
    nearest["nearest_school_m"],
    bins=[0, 100, 250, 500, float("inf")],
    labels=[
        "0-100m",
        "100-250m",
        "250-500m",
        "500m+"
    ],
    include_lowest=True
)

print(nearest["school_proximity"].value_counts())

school_proximity
100-250m    78
250-500m    52
0-100m      28
500m+        0
Name: count, dtype: int64


In [23]:
detectors_m = detectors_geo.to_crs("EPSG:2056")
schools_m = schools.to_crs("EPSG:2056").copy()

# Assign each school feature a unique identifier
schools_m = schools_m.reset_index(drop=True)
schools_m["school_id"] = schools_m.index

for radius in [100, 250, 500]:
    # Create buffers around detectors
    buffers = detectors_m[["detid", "geometry"]].copy()
    buffers["geometry"] = buffers.geometry.buffer(radius)

    # Find schools within each buffer
    matched = gpd.sjoin(
        buffers,
        schools_m[["school_id", "geometry"]],
        how="left",
        predicate="intersects"
    )

    # Count schools per detector
    counts = (
        matched.groupby("detid")["school_id"]
        .nunique()
    )

    detectors_m[f"schools_{radius}m"] = (
        detectors_m["detid"]
        .map(counts)
        .fillna(0)
        .astype(int)
    )

# Verify results
display(
    detectors_m[
        ["schools_100m", "schools_250m", "schools_500m"]
    ].describe()
)

print("Unique detectors:", detectors_m["detid"].nunique())

,schools_100m,schools_250m,schools_500m
count,158.000000,158.000000,158.000000
mean,0.177215,1.107595,4.462025
std,0.383065,1.038037,2.306557
min,0.000000,0.000000,1.000000
25%,0.000000,0.000000,3.000000
50%,0.000000,1.000000,4.000000
75%,0.000000,2.000000,6.000000
max,1.000000,3.000000,9.000000


Unique detectors: 158


In [25]:
con = duckdb.connect()

traffic = con.execute("""
    SELECT
        detid,
        day,
        interval,
        flow,
        occ
    FROM read_parquet(?)
    WHERE error IS NULL
      AND flow IS NOT NULL
      AND interval >= 0
      AND interval < 86400
""", [str(DATA_DIR / "luzern_raw.parquet")]).df()

traffic["hour"] = traffic["interval"] // 3600
traffic["day"] = pd.to_datetime(traffic["day"])
traffic["weekday"] = traffic["day"].dt.dayofweek

print("Measurements:", len(traffic))

Measurements: 17781141


In [26]:
# Weekdays only
traffic = traffic[traffic["weekday"] < 5].copy()

def assign_period(hour):
    if 7 <= hour < 9:school_features = detectors_m[
    ["detid", "schools_100m", "schools_250m", "schools_500m"]
].copy()

analysis = detector_flow.merge(
    school_features,
    on="detid",
    how="inner",
    validate="many_to_one"
)

display(analysis.head())
        return "morning"
    if 11 <= hour < 14:
        return "midday"
    if 15 <= hour < 18:
        return "afternoon"
    return None

traffic["period"] = traffic["hour"].apply(assign_period)
traffic = traffic.dropna(subset=["period"])

detector_flow = (
    traffic.groupby(["detid", "period"])["flow"]
    .mean()
    .reset_index(name="mean_flow")
)

display(detector_flow.head())

,detid,period,mean_flow
0,ig11FD100_D3,afternoon,329.424316
1,ig11FD100_D3,midday,316.342040
2,ig11FD100_D3,morning,309.591885
3,ig11FD100_D31,afternoon,589.004806
4,ig11FD100_D31,midday,523.996639


In [27]:
school_features = detectors_m[
    ["detid", "schools_100m", "schools_250m", "schools_500m"]
].copy()

analysis = detector_flow.merge(
    school_features,
    on="detid",
    how="inner",
    validate="many_to_one"
)

display(analysis.head())

,detid,period,mean_flow,schools_100m,schools_250m,schools_500m
0,ig11FD100_D3,afternoon,329.424316,1,2,5
1,ig11FD100_D3,midday,316.342040,1,2,5
2,ig11FD100_D3,morning,309.591885,1,2,5
3,ig11FD100_D31,afternoon,589.004806,0,1,9
4,ig11FD100_D31,midday,523.996639,0,1,9


In [28]:
for period in ["morning", "midday", "afternoon"]:
    subset = analysis[analysis["period"] == period]

    print(f"\n{period.upper()}")

    for feature in ["schools_250m", "schools_500m"]:
        pearson = subset[feature].corr(
            subset["mean_flow"], method="pearson"
        )
        spearman = subset[feature].corr(
            subset["mean_flow"], method="spearman"
        )

        print(
            f"{feature}: "
            f"Pearson={pearson:.3f}, "
            f"Spearman={spearman:.3f}"
        )


MORNING
schools_250m: Pearson=0.034, Spearman=-0.003
schools_500m: Pearson=-0.011, Spearman=-0.024

MIDDAY
schools_250m: Pearson=0.154, Spearman=0.117
schools_500m: Pearson=0.046, Spearman=0.038

AFTERNOON
schools_250m: Pearson=0.135, Spearman=0.114
schools_500m: Pearson=0.012, Spearman=0.002


## Conclusion - School Proximity and Traffic Flow

We investigated whether the proximity and density of schools influence traffic flow in Luzern, Switzerland.

Using OpenStreetMap, we identified **48 school-tagged locations** and mapped them against **158 traffic detectors** from the UTD19 dataset. For each detector, we calculated the distance to the nearest school and the number of schools within 100 m, 250 m, and 500 m.

We then analyzed approximately **17.8 million traffic measurements**, using provisionally unflagged observations, and calculated Pearson and Spearman correlations between school density and average traffic flow during weekday morning, midday, and afternoon periods.

### Results

| Period | Pearson (250 m) | Spearman (250 m) | Pearson (500 m) | Spearman (500 m) |
|---|---:|---:|---:|---:|
| Morning | 0.034 | -0.003 | -0.011 | -0.024 |
| Midday | 0.154 | 0.117 | 0.046 | 0.038 |
| Afternoon | 0.135 | 0.114 | 0.012 | 0.002 |

### Findings

- All 158 detectors were located within 500 meters of at least one mapped school.
- Correlations between school density and average traffic flow were consistently weak or negligible.
- The highest observed Pearson correlation was **0.154**, indicating only a very weak positive linear relationship.
- Neither Pearson nor Spearman correlation demonstrated a meaningful association between school density and average traffic flow.

### Final Conclusion

**The analysis did not identify a meaningful correlation between school density and average traffic flow in Luzern.**

Based on these findings, school proximity alone does not appear to be a strong predictor of average traffic flow in the examined dataset.

This conclusion is limited to the tested proximity thresholds and time periods. Current OpenStreetMap school locations were used alongside 2015 traffic measurements, assuming that the geographical distribution of schools remained sufficiently stable.

We will therefore not prioritize school-density features in the initial TrafficTwin forecasting model and will continue investigating other potentially relevant traffic predictors.